# Semantic Counting in Vector Databases — Demo

This notebook walks through the full pipeline:

1. **Load** the Banking77 dataset (customer intent queries)
2. **Embed** sentences with `all-MiniLM-L6-v2`
3. **Cluster** via HDBSCAN
4. **Summarise** clusters with Google Colab's built-in LLM
5. **Query** — run `semantic_count(*)` with yes/no filtering **and** similarity scoring (0–1)
6. **Ranked output** — matched sentences ranked by similarity score, saved to `.txt` files
7. **Baseline** — compare against an embedding-only approach

No API keys required — uses `google.colab.ai.generate_text()` for all LLM calls.

## 0. Setup — Clone Repo & Install Dependencies

In [ ]:
!git clone https://github.com/donggunkwak/Semantic-Count.git
%cd Semantic-Count
!pip install -q -r requirements.txt

In [ ]:
import sys, os

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

### Verify Colab AI is available

In [ ]:
from google.colab import ai

test = ai.generate_text("Say hello in one word.")
print(f"Colab AI test: {test}")

## 1. Load Banking77 Sentences

In [ ]:
from src.data_loader import load_banking77_sentences

sentences = load_banking77_sentences()
print(f"Total unique sentences: {len(sentences)}")
print("\nFirst 5 sentences:")
for s in sentences[:5]:
    print(f"  \u2022 {s}")

## 2. Generate Embeddings

In [ ]:
from src.embeddings import generate_embeddings

embeddings = generate_embeddings(sentences)
print(f"Embeddings shape: {embeddings.shape}")

## 3. Cluster with HDBSCAN

In [ ]:
from src.clustering import cluster_embeddings, get_cluster_members

labels = cluster_embeddings(embeddings)

cluster_ids = sorted(set(labels) - {-1})
members = get_cluster_members(labels)

print(f"\nNumber of clusters: {len(cluster_ids)}")
print(f"Noise points: {labels.count(-1)}")
print(f"\nCluster sizes (top 10):")
sizes = sorted(((cid, len(members[cid])) for cid in cluster_ids), key=lambda x: -x[1])
for cid, size in sizes[:10]:
    print(f"  Cluster {cid}: {size} sentences")

## 4. Summarise Clusters (LLM)

This step uses Google Colab's built-in LLM to generate a short semantic summary for each cluster. Results are cached to `data/cluster_summaries.json`.

In [ ]:
from src.summarizer import summarize_clusters

summaries = summarize_clusters(sentences, labels)

print(f"\nCluster summaries ({len(summaries)} clusters):")
print("-" * 60)
for cid in sorted(summaries, key=int):
    print(f"  Cluster {cid}: {summaries[cid]}")
    print()

## 5. Semantic Counting — Example Queries

We run five example queries. For each query the engine:
1. Embeds the query
2. Finds the top-K nearest clusters
3. Filters clusters with the LLM
4. Checks individual documents — yes/no **plus** a similarity score (0–1, where 1.0 = exact match)
5. Ranks all matched sentences by score (highest → lowest)
6. Returns the semantic count and saves ranked results to a `.txt` file

In [ ]:
from src.query_engine import semantic_count

queries = [
    "I dropped my credit card in the river.",
    "I paid $10 to get an A in this class, but I was charged $100.",
]

results = {}
for q in queries:
    print("=" * 60)
    print(f'Query: "{q}"')
    print("=" * 60)
    result = semantic_count(
        q, sentences, embeddings, labels, summaries,
        save_path=None,
        save_ranked_txt=True,
    )
    results[q] = result
    print()

### Display Results

In [ ]:
import pandas as pd

rows = []
for q, r in results.items():
    scores = [e["score"] for e in r.scored_sentences]
    avg_score = sum(scores) / len(scores) if scores else 0.0
    rows.append({
        "Query": q,
        "Clusters Retrieved": r.clusters_retrieved,
        "Clusters After LLM Filter": r.clusters_after_llm_filter,
        "Documents Checked": r.documents_checked,
        "semantic_count(*)": r.semantic_count,
        "Avg Similarity": round(avg_score, 4),
    })

df = pd.DataFrame(rows)
display(df)

In [ ]:
for q, r in results.items():
    print(f'\n--- Cluster details for "{q}" ---')
    for c in r.cluster_details:
        status = "KEPT" if c["relevant"] else "FILTERED"
        print(f"  [{status}] Cluster {c['cluster_id']}: {c['summary']}")

    ranked = r.ranked_sentences
    print(f"\n  Matched sentences ranked by similarity ({r.semantic_count} total):")
    for entry in ranked[:10]:
        print(f"    \u2022 [score={entry['score']:.4f}] {entry['sentence']}")
    if r.semantic_count > 10:
        print(f"    ... and {r.semantic_count - 10} more (see .txt file for full list)")

## Summary

The LLM-based semantic counting pipeline:
- Uses clustering to **prune the search space** (only check relevant clusters)
- Uses LLM calls at two levels: **cluster filtering** (cheap) and **document checking with similarity scoring** (precise)
- Each matched document receives a **similarity score (0–1)**, where 1.0 means the sentence is essentially identical to the query
- Matched sentences are **ranked by score** (highest → lowest) and saved to `.txt` files in `outputs/`
- Produces more semantically accurate counts than a simple cosine threshold

The baseline is fast and requires no LLM calls, but cannot capture nuanced semantic distinctions.